# **ASR EVALUATION**

## **Imports**

In [2]:
import sys
sys.path.append("../scripts/asr")

from asr_metrics import compute_asr_metrics

import pandas as pd
import re


## **1. Baseline Whisper WER and CER Evaluation**

### **HINDI**

In [3]:
# Basic Cleaning Function
def normalize_text(text):
    text = str(text)

    # lowercase
    text = text.lower()

    # remove punctuation
    text = re.sub(r'[^\w\s\u0900-\u097F]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [4]:
df = pd.read_csv("../experiments/asr/whisper_baseline/hindi_whisper_train.csv")

# remove missing predictions
df = df.dropna(subset=["prediction"])

# ensure strings
df["prediction"] = df["prediction"].astype(str)
df["text"] = df["text"].astype(str)

print("After cleaning:", len(df))

# Apply normalization
df["prediction"] = df["prediction"].apply(normalize_text)
df["text"] = df["text"].apply(normalize_text)

# compute metrics
metrics = compute_asr_metrics(df["text"].to_list(), df["prediction"].to_list())

print(metrics)

After cleaning: 36684
{'WER': 1.3899921197793539, 'CER': 1.2571369349844899}


*Observation:*

- High WER (1.389) and CER (1.257), mainly due to multilingual drift and script inconsistency

##### **Filtered Hindi-only WER and CER(for linguistic evaluation)**

In [5]:
# -------------------------
# FILTER: ONLY HINDI PREDICTIONS
# -------------------------
def has_hindi(text):
    return bool(re.search(r'[\u0900-\u097F]', text))

df_hindi = df[df["prediction"].apply(has_hindi)].copy()

print("Total samples:", len(df))
print("Hindi-only samples:", len(df_hindi))
print("\n")
metrics_hindi = compute_asr_metrics(df_hindi["text"].to_list(), df_hindi["prediction"].to_list())
print(metrics_hindi)

Total samples: 36684
Hindi-only samples: 16937


{'WER': 1.5690455919752884, 'CER': 1.429947437996622}


*Obervation:*

- High WER(1.569) and CER (1.429), indicating high error rates.

##### **Final Sanity Check of Data**

In [6]:
for i in range(10):
    print("REF:", df["text"].iloc[i])
    print("PRED:", df["prediction"].iloc[i])
    print("-"*50)

REF: और ऐसे ही बड़ी खबरों के लिए सुनते रहे मधुवनी मोबाइल वाणी धन्यवाद
PRED: اور ایسے ہی بڑی خبروکیل سنتے رہے مدوانی موائلوانی دن نواض
--------------------------------------------------
REF: के लिए पचीस जुलाई के बाद विश्वविद्यालय के वेबसाइट ऐसी परवेश पात्र डाउनलोड कर सकते हैं बिहार राज्य के मधुवनी जिला ऐसी रामानंद जी
PRED: पाराट देगे मेजला से नामनादाणई जी سے پر vespatra download کر سکھے vihar raji ke madhwani jila se ramanand ji
--------------------------------------------------
REF: पी जी जी विनामंकन प्रक्रिया की जानकारी दी हैं ललिथ नारायण मिथिला विश्वविद्द्यालय ने पी जी जी नामांकन की प्रक्रिया शुरू हो गए है नामांकन के लिए
PRED: pgmnamangkan prakaria ki jankari dih lalitnaran mithala biswavidale med pgmnamangamangkan ki parkiria suruh ogai hain namangkan kelia park
--------------------------------------------------
REF: बिहार राज्य के मधुबनी जिला से मनोज कर्नल ने मोबाइल वाणी के माध्यम ऐसी बताया की सुरक्षा कर्मियों ने तीन बोरियोंसे भरे
PRED: یہار راجکے مدوانی جیلاخ سے منوج کرنے مووائیل و

***Final Obervation for Hindi ASR:***

Although Whisper captured phonetic content reasonably well,
it frequently produced outputs in Urdu or Romanized script,
leading to poor WER scores and reduced usability for Hindi applications.

---

### **English**

In [7]:
def normalize_english(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
df_english = pd.read_csv("../experiments/asr/whisper_baseline/english_whisper_train.csv")
print("Total samples:", len(df_english))
print("Missing predictions:", df_english["prediction"].isna().sum())

# Remove missing predictions
df_english = df_english.dropna(subset=["prediction"])

# Ensure strings
df_english["prediction"] = df_english["prediction"].astype(str)
df_english["text"] = df_english["text"].astype(str)
print("After cleaning:", len(df_english))

# Apply normalization
df_english["prediction"] = df_english["prediction"].apply(normalize_english)
df_english["text"] = df_english["text"].apply(normalize_english)

# Compute metrics
metrics_english = compute_asr_metrics(df_english["text"].to_list(), df_english["prediction"].to_list())
print(metrics_english)

Total samples: 28539
Missing predictions: 0
After cleaning: 28539
{'WER': 0.06448736038040563, 'CER': 0.028996073755301343}


*Oberservation:*

- Very low WER(0.064) and CER (0.028), indicating very high accuracy for english dataset.

## ***Final Interpretation for baseline Whisper model***

The Whisper base model demonstrated strong performance on English speech recognition,
achieving a low WER of 0.064 and CER of 0.029. However, when applied to Hindi audio,
the model exhibited significantly degraded performance, with WER exceeding 1.4.

Qualitative analysis revealed that the model frequently produced outputs in Urdu
script or Romanized Hindi instead of Devanagari. This script inconsistency led to
high error rates despite partially correct phonetic transcription.

These findings indicate that while Whisper is effective for high-resource languages
like English, it struggles with script fidelity in multilingual and low-resource
language settings such as Hindi.

| Language | WER   | CER   | Observation                  |
|----------|-------|-------|------------------------------|
| English  | 0.064 | 0.029 | High accuracy ✔              |
| Hindi    | 1.44  | 1.31  | Very poor performance ❌     |

---
---

## **2. Wav2Vec2 Model Evaluation**

### **Hindi**

In [9]:
df = pd.read_csv("../experiments/asr/wav2vec2/hindi_predictions.csv")

# remove missing predictions
df = df.dropna(subset=["prediction"])

# ensure strings
df["prediction"] = df["prediction"].astype(str)
df["text"] = df["text"].astype(str)

print("After cleaning:", len(df))

# Apply normalization
df["prediction"] = df["prediction"].apply(normalize_text)
df["text"] = df["text"].apply(normalize_text)

# compute metrics
metrics = compute_asr_metrics(df["text"].to_list(), df["prediction"].to_list())

print(metrics)

After cleaning: 1882
{'WER': 0.39788591522604283, 'CER': 0.1746856642164374}


*Observation:*

- The wav2vec2 model uses AI4Bharat model as a base, thus giving a very good WER of 0.397 and CER of 0.174.
- High phonetic and meaning preservation seen.
- Will require an LM to better the results.

### **ENGLISH**

In [10]:
df_english = pd.read_csv("../experiments/asr/whisper_baseline/english_whisper_train.csv")
print("Total samples:", len(df_english))
print("Missing predictions:", df_english["prediction"].isna().sum())

# Remove missing predictions
df_english = df_english.dropna(subset=["prediction"])

# Ensure strings
df_english["prediction"] = df_english["prediction"].astype(str)
df_english["text"] = df_english["text"].astype(str)
print("After cleaning:", len(df_english))

# Apply normalization
df_english["prediction"] = df_english["prediction"].apply(normalize_english)
df_english["text"] = df_english["text"].apply(normalize_english)

# Compute metrics
metrics_english = compute_asr_metrics(df_english["text"].to_list(), df_english["prediction"].to_list())
print(metrics_english)

Total samples: 28539
Missing predictions: 0
After cleaning: 28539
{'WER': 0.06448736038040563, 'CER': 0.028996073755301343}


*Oberservation:*

- Very low WER(0.064) and CER (0.028), indicating very high accuracy for english dataset.
- It uses Wav2Vec2-base-940h model.

## ***Final Interpretation for Wav2Vec2 models***

The experimental results demonstrate that the CTC-based Wav2Vec2 model is highly effective for speech recognition across both high-resource and low-resource languages. For English, the model achieves a low WER of 6.4% and CER of 2.9%, indicating near production-level accuracy due to extensive pretraining on large-scale datasets. In contrast, the Hindi model records a higher WER of 39% but a relatively lower CER of 17%, revealing that while exact word matching is weaker, the model successfully captures phonetic and semantic content. Most errors in Hindi are attributable to spelling variations and script complexities rather than loss of meaning. This highlights a key characteristic of CTC-based models: they prioritize phonetic alignment over strict lexical accuracy. Consequently, despite higher WER, the Hindi ASR output remains semantically meaningful and suitable for downstream tasks such as summarization. Overall, these results validate the robustness and adaptability of Wav2Vec2 with CTC for multilingual speech recognition, especially in real-world scenarios where preserving meaning is more critical than perfect transcription.

| Language | WER   | CER   | Observation                  |
|----------|-------|-------|------------------------------|
| English  | 0.064 | 0.029 | High accuracy ✔              |
| Hindi    | 0.39  | 0.17  | Great performance   ✔   |

---
---